In [1]:
#IMPORTS
import os
import sys
import arrow
import numpy as np
import pandas as pd
from math import pi
import matplotlib.pyplot as plt
import matplotlib.dates as dates
import seaborn as sns
from datetime import date, datetime, timedelta
from numpy import nan_to_num
from matplotlib.colors import LogNorm
from matplotlib.ticker import ScalarFormatter
from scipy.optimize import curve_fit
import scipy.odr as odr

MODULE 0: Prepare May dataset files to fit Module A, C, and D

Part 1: Fix Date and UTC columns for May dataset files.
Run this regardless if you want to run Modules A, C, or D
Also, removes rows with atleast one NaN or empty columns

In [2]:
#FIX DATE and UTC columns in NASA_restricted.csv
FILE_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\restricted_combined\master_restricted.csv"

# Your column list
column_list = ["Latitude", "Longitude", "Altitude", "Temperature", "Rel_humidity", "Pressure", 'BC_Mass', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5', 'bin6', 'bin7', 'bin8', 'bin9', 'bin10', 'bin11', 'bin12', 'bin13', 'bin14', 'bin15', 'Sc450_total', 'Sc550_total', 'Sc700_total', 'Abs470_total', 'Abs532_total', 'Abs660_total']

#changes UTC and Date columns to datetime format
def add_datetime_column(df):
    """
    Takes a DataFrame with 'Date' (YYYYMMDD) and 'UTC' (seconds after midnight)
    and adds a new 'datetime' column in MM/DD/YYYY HH:MM:SS format.
    """
    def convert_to_datetime(row):
        # Parse the Date
        date_str = str(row['Date'])
        date_obj = datetime.strptime(date_str, "%Y%m%d")
        # Add seconds after midnight
        full_datetime = date_obj + timedelta(seconds=row['UTC'])
        # Format to MM/DD/YYYY HH:MM:SS
        return full_datetime.strftime("%m/%d/%Y %H:%M:%S")
    
    # Create the datetime column separately
    datetime_series = df.apply(convert_to_datetime, axis=1)

    # Find where 'UTC' is located
    utc_index = df.columns.get_loc('UTC')
    
    # Insert the new column before 'UTC'
    df.insert(utc_index, 'datetime', datetime_series)
    return df

def check_invalid_values(df, columns_to_check):
    """
    Check for NaN, inf, and -inf values in specified columns
    """
    print(f"Checking for invalid values (NaN, inf, -inf) in {len(columns_to_check)} specified columns...")
    
    invalid_summary = {}
    
    for col in columns_to_check:
        if col not in df.columns:
            print(f"WARNING: Column '{col}' not found in DataFrame!")
            continue
            
        # Check for different types of invalid values
        nan_count = df[col].isnull().sum()
        inf_count = np.isinf(df[col]).sum()
        neginf_count = np.isneginf(df[col]).sum()
        total_invalid = nan_count + inf_count + neginf_count
        
        if total_invalid > 0:
            invalid_summary[col] = {
                'nan': nan_count,
                'inf': inf_count,
                'neginf': neginf_count,
                'total': total_invalid
            }
    
    return invalid_summary

df_may = pd.read_csv(FILE_PATH)
df_may = add_datetime_column(df_may)

# Diagnostic prints before filtering
print("=" * 80)
print("BEFORE FILTERING:")
print("=" * 80)
print(f"Total rows: {len(df_may)}")
print(f"Total columns: {len(df_may.columns)}")

# Check which columns from column_list actually exist in the DataFrame
existing_columns = [col for col in column_list if col in df_may.columns]
missing_columns = [col for col in column_list if col not in df_may.columns]

print(f"\nColumn List Analysis:")
print(f"Columns to check: {len(column_list)}")
print(f"Columns found in DataFrame: {len(existing_columns)}")
print(f"Missing columns: {len(missing_columns)}")

if missing_columns:
    print(f"Missing columns: {missing_columns}")

# Check for invalid values in the specified columns only
invalid_summary = check_invalid_values(df_may, existing_columns)

print(f"\nInvalid Values Summary for Specified Columns:")
if invalid_summary:
    for col, counts in invalid_summary.items():
        print(f"{col}:")
        print(f"  NaN: {counts['nan']}")
        print(f"  inf: {counts['inf']}")
        print(f"  -inf: {counts['neginf']}")
        print(f"  Total invalid: {counts['total']}")
        
        # Show Organization-Campaign combinations for this column
        invalid_mask = df_may[col].isnull() | np.isinf(df_may[col])
        if invalid_mask.sum() > 0:
            invalid_rows = df_may[invalid_mask]
            org_campaign_combinations = invalid_rows.groupby(['Organization', 'Campaign']).size().reset_index(name='count')
            print(f"  Organization-Campaign combinations with invalid values:")
            for _, row in org_campaign_combinations.iterrows():
                print(f"    {row['Organization']}-{row['Campaign']}: {row['count']} invalid values")
        print()
else:
    print("No invalid values found in specified columns!")

# Create mask for rows with invalid values in ANY of the specified columns
print("Creating filter mask...")
invalid_mask = pd.Series(False, index=df_may.index)

for col in existing_columns:
    # Add to mask: NaN, inf, or -inf values
    col_invalid = df_may[col].isnull() | np.isinf(df_may[col])
    invalid_mask = invalid_mask | col_invalid
    
    if col_invalid.sum() > 0:
        print(f"  {col}: {col_invalid.sum()} invalid values")

# Count rows that have invalid values in ANY of the specified columns
rows_with_invalid_values = invalid_mask.sum()
print(f"\nRows with invalid values in ANY specified column: {rows_with_invalid_values}")

# Filter out rows where ANY specified column has invalid values
df_may_filtered = df_may[~invalid_mask].copy()

# Diagnostic prints after filtering
print("\n" + "=" * 80)
print("AFTER FILTERING:")
print("=" * 80)
print(f"Total rows: {len(df_may_filtered)}")
print(f"Rows removed: {len(df_may) - len(df_may_filtered)}")
print(f"Percentage of data retained: {(len(df_may_filtered) / len(df_may)) * 100:.2f}%")

# Verify no invalid values remain in specified columns
print(f"\nVerification - checking specified columns for remaining invalid values:")
remaining_invalid = check_invalid_values(df_may_filtered, existing_columns)

if remaining_invalid:
    print("WARNING: Some invalid values still remain!")
    for col, counts in remaining_invalid.items():
        print(f"{col}: {counts['total']} invalid values")
else:
    print("✓ No invalid values remaining in specified columns!")

# Check other columns (not in column_list) for reference
other_columns = [col for col in df_may_filtered.columns if col not in column_list]
other_invalid_count = 0
for col in other_columns:
    if df_may_filtered[col].dtype in ['float64', 'float32', 'int64', 'int32']:
        col_invalid = df_may_filtered[col].isnull().sum() + np.isinf(df_may_filtered[col]).sum()
        other_invalid_count += col_invalid

print(f"\nInvalid values in OTHER columns (not filtered): {other_invalid_count}")

# Show final dataset composition
print(f"\nFinal Dataset Composition:")
final_org_campaign = df_may_filtered.groupby(['Organization', 'Campaign']).size().reset_index(name='count')
for _, row in final_org_campaign.iterrows():
    print(f"  {row['Organization']}-{row['Campaign']}: {row['count']} rows")

# Save the filtered dataset
output_path = rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleA\master_restricted_datetimefixed.csv"
df_may_filtered.to_csv(output_path, index=False)
print(f"\nFiltered dataset saved successfully to:")
print(f"{output_path}")
print(f"Final dataset shape: {df_may_filtered.shape}")

BEFORE FILTERING:
Total rows: 4737635
Total columns: 33

Column List Analysis:
Columns to check: 28
Columns found in DataFrame: 28
Missing columns: 0
Checking for invalid values (NaN, inf, -inf) in 28 specified columns...

Invalid Values Summary for Specified Columns:
Latitude:
  NaN: 72880
  inf: 0
  -inf: 0
  Total invalid: 72880
  Organization-Campaign combinations with invalid values:
    NASA-DISCOVERAQ-California: 14 invalid values
    NASA-DISCOVERAQ-DC: 10 invalid values
    NASA-FIREXAQ: 29930 invalid values
    NASA-NAAMES(2015): 34 invalid values
    NASA-NAAMES(2016): 11 invalid values
    NASA-SEAC4RS: 42881 invalid values

Longitude:
  NaN: 72879
  inf: 0
  -inf: 0
  Total invalid: 72879
  Organization-Campaign combinations with invalid values:
    NASA-DISCOVERAQ-California: 13 invalid values
    NASA-DISCOVERAQ-DC: 10 invalid values
    NASA-FIREXAQ: 29930 invalid values
    NASA-NAAMES(2015): 34 invalid values
    NASA-NAAMES(2016): 11 invalid values
    NASA-SEAC4RS: 

Part 2: Generate size_dist_number.csv and size_dist_diameter.csv for MODULE A

In [3]:
#Generate size_dist_number.csv and size_dist_diameter.csv for MODULE A
AEROSOL_DIAMS = [150, 169.8, 192.1, 217.5, 246.1, 278.6, 315.3, 356.8, 403.9, 457.1, 517.3, 585.5, 662.7, 750]

#read newly date time fixed NASA file
#df_may = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleA\test.csv")
df_may = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleA\master_restricted_datetimefixed.csv")

#POPULATE datetime.csv
df_datetime = df_may[['datetime']].copy()
df_datetime = df_datetime.rename(columns={'datetime': 'datetime_all'})
df_datetime.to_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleA\datetime.csv", index=False)

#POPULATE size_dist_diameter_input.csv
df = pd.DataFrame({'diameter': AEROSOL_DIAMS})

# Save to CSV
df.to_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleA\size_dist_diameter_input.csv", index=False)

#POPULATE size_dist_input.csv
# Select only columns that start with "bin"
bin_columns = [col for col in df_may.columns if col.startswith('bin')]

# Create a new DataFrame with only the bin columns
df_bins = df_may[bin_columns]

# Drop the last bin column
df_bins = df_bins.iloc[:, :-1]

# Transpose the DataFrame (rows = diameters, columns = time steps)
df_bins_transposed = df_bins.transpose()

# Reset the index so that "bin1", "bin2", etc., are not saved as extra rows/columns
df_bins_transposed = df_bins_transposed.reset_index(drop=True)

df_bins_transposed.to_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleA\size_dist_input.csv", index=False)

Part 3: Generate Bscat_raw.csv and Babs_raw.csv for Module C

In [4]:
# Load the full dataframe
#df_may = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleA\test.csv")
# Load full dataframe
df_may = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleA\master_restricted_datetimefixed.csv")

# --- Scattering: match data correctly ---
# Sc450 (blue), Sc550 (green), Sc700 (red)
df_scat = df_may[["Sc700_total", "Sc550_total", "Sc450_total"]].copy()
df_scat.columns = ["scat_red", "scat_green", "scat_blue"]
df_scat.to_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleC\Bscat_raw.csv", index=False)

# --- Absorption: match data correctly ---
# Abs470 (blue), Abs532 (green), Abs660 (red)
df_abs = df_may[["Abs660_total", "Abs532_total", "Abs470_total"]].copy()
df_abs.columns = ["abs_red", "abs_green", "abs_blue"]
df_abs.to_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleC\Babs_raw.csv", index=False)

Part 4: Generate BC_Mass.csv for MODULE D

Note: To run MODULE D, you need Output_Module_C.csv which you get by running MODULE C

In [5]:
#read entire may dataset
df_may = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleA\master_restricted_datetimefixed.csv")

#extract BC_Mass column
BC_Mass = df_may['BC_Mass'].copy()

#Output only BC_Mass column to a new CSV file
BC_Mass.to_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleD\BC_mass.csv", index = False)